# Real-Time Content Moderation
Fine-tunes DistilBERT on labeled posts (Wikipedia comments + real tweets) to flag toxic
content, then optimizes it with ONNX Runtime for fast inference.



In [9]:
# Remove torchvision — we don't need it, and Colab's version conflicts with `datasets`
!pip uninstall -y torchvision -q


**STOP — Restart the runtime now.**
Runtime → Restart session → then continue with the cell below.
(Uninstalling alone doesn't help until Python's memory is cleared.)


In [1]:
# Run this AFTER restarting the session.
# Pinned versions known to work together — avoids the optimum/diffusers/transformers
# version conflicts hit during development. We deliberately do NOT install `optimum`.
!pip install -q "transformers==4.46.3" "accelerate==1.1.1" "datasets==3.1.0" "evaluate==0.4.3" \
    "scikit-learn" "onnx==1.17.0" "onnxruntime==1.19.2" "onnxscript" -q


Restart the session ONE more time after this install finishes, then continue below.

In [1]:
import pandas as pd
import numpy as np
import torch
import time
import os
import json

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/toxicity-project'
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Drive checkpoint folder:", DRIVE_DIR)


Torch: 2.11.0+cu128
CUDA available: True
Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive checkpoint folder: /content/drive/MyDrive/toxicity-project


## 2. Load the Jigsaw dataset (your uploaded file)

Upload `train.csv` (Jigsaw Toxic Comment Classification format:
`id, comment_text, toxic, severe_toxic, obscene, threat, insult, identity_hate`).


In [2]:
from google.colab import files

uploaded = files.upload()  # select train.csv
DATA_PATH = list(uploaded.keys())[0]


Saving train.csv to train.csv


In [3]:
jigsaw_df = pd.read_csv(DATA_PATH)
print("Jigsaw raw shape:", jigsaw_df.shape)

LABEL_COLS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# Collapse multi-label -> single binary target (any category = toxic).
# Real-time moderation needs one flag/don't-flag decision, not a category breakdown.
jigsaw_df['is_toxic'] = (jigsaw_df[LABEL_COLS].sum(axis=1) > 0).astype(int)
jigsaw_df = jigsaw_df[['comment_text', 'is_toxic']].dropna().reset_index(drop=True)
jigsaw_df['comment_text'] = jigsaw_df['comment_text'].astype(str).str.slice(0, 2000)
jigsaw_df['source'] = 'jigsaw_comments'

print(jigsaw_df['is_toxic'].value_counts(normalize=True))
jigsaw_df.head()


Jigsaw raw shape: (159571, 8)
is_toxic
0    0.898321
1    0.101679
Name: proportion, dtype: float64


,comment_text,is_toxic,source
0,Explanation\nWhy the edits made under my usern...,0,jigsaw_comments
1,D'aww! He matches this background colour I'm s...,0,jigsaw_comments
2,"Hey man, I'm really not trying to edit war. It...",0,jigsaw_comments
3,"""\nMore\nI can't make any real suggestions on ...",0,jigsaw_comments
4,"You, sir, are my hero. Any chance you remember...",0,jigsaw_comments


## 3. Load the Twitter dataset (real social media posts)

Downloaded automatically — no upload needed. This is what makes "social media posts" in
the resume bullet accurate; Jigsaw alone is Wikipedia comments, not posts.


In [4]:
from datasets import load_dataset

# Note: uses the namespaced repo id — the old bare name was deprecated by HF Hub.
twitter_raw = load_dataset("tweets-hate-speech-detection/tweets_hate_speech_detection")
twitter_df = twitter_raw['train'].to_pandas()

twitter_df = twitter_df.rename(columns={'tweet': 'comment_text', 'label': 'is_toxic'})
twitter_df = twitter_df[['comment_text', 'is_toxic']].dropna().reset_index(drop=True)
twitter_df['comment_text'] = twitter_df['comment_text'].astype(str).str.slice(0, 2000)
twitter_df['source'] = 'twitter_posts'

print("Twitter shape:", twitter_df.shape)
print(twitter_df['is_toxic'].value_counts(normalize=True))
twitter_df.head()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.07M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/31962 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17197 [00:00<?, ? examples/s]

Twitter shape: (31962, 3)
is_toxic
0    0.929854
1    0.070146
Name: proportion, dtype: float64


,comment_text,is_toxic,source
0,@user when a father is dysfunctional and is so...,0,twitter_posts
1,@user @user thanks for #lyft credit i can't us...,0,twitter_posts
2,bihday your majesty,0,twitter_posts
3,#model i love u take with u all the time in ...,0,twitter_posts
4,factsguide: society now #motivation,0,twitter_posts


## 4. Balance the two sources equally and combine

Jigsaw (159K) would outnumber Twitter (32K) roughly 5:1 in a raw merge, leaving the model
mostly learning Wikipedia-comment patterns. We downsample Jigsaw to Twitter's size,
preserving each source's own toxic/clean ratio, so the combined dataset is genuinely
balanced between the two platforms — not dominated by one.


In [5]:
def stratified_downsample(source_df, n, label_col='is_toxic', seed=42):
    """Downsample a df to n rows, keeping its own toxic/clean ratio intact."""
    frac = n / len(source_df)
    return (
        source_df.groupby(label_col, group_keys=False)
        .apply(lambda g: g.sample(frac=frac, random_state=seed))
        .reset_index(drop=True)
    )

target_size = min(len(jigsaw_df), len(twitter_df))

jigsaw_balanced = stratified_downsample(jigsaw_df, target_size)
twitter_balanced = twitter_df

df = pd.concat([jigsaw_balanced, twitter_balanced], ignore_index=True)
df = df.drop_duplicates(subset='comment_text').sample(frac=1, random_state=42).reset_index(drop=True)

print("Combined shape:", df.shape)
print(df['source'].value_counts())
print()
print("Combined label balance:")
print(df['is_toxic'].value_counts(normalize=True))

df.to_csv(os.path.join(DRIVE_DIR, 'combined_dataset.csv'), index=False)
print("Saved combined dataset to Drive as a backup.")


/tmp/ipykernel_3750/1582985204.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(frac=frac, random_state=seed))


Combined shape: (61486, 3)
source
jigsaw_comments    31959
twitter_posts      29527
Name: count, dtype: int64

Combined label balance:
is_toxic
0    0.914485
1    0.085515
Name: proportion, dtype: float64
Saved combined dataset to Drive as a backup.


## 5. Train / validation / test split (stratified — labels are imbalanced ~10%)

In [6]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['is_toxic'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['is_toxic'], random_state=42)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)

# Save the exact test set to Drive — needed later to re-evaluate without retraining
test_df.to_csv(os.path.join(DRIVE_DIR, 'test_set.csv'), index=False)


Train: (49188, 3) Val: (6149, 3) Test: (6149, 3)


## 6. Tokenize

In [7]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(pdf):
    return Dataset.from_pandas(pdf.rename(columns={'is_toxic': 'label'})[['comment_text', 'label']].reset_index(drop=True))

train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)
test_ds = to_hf_dataset(test_df)

def tokenize_fn(batch):
    return tokenizer(batch['comment_text'], truncation=True, padding='max_length', max_length=128)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

# DistilBERT doesn't accept token_type_ids — drop it here so it never causes problems downstream
for ds in (train_ds, val_ds, test_ds):
    if 'token_type_ids' in ds.column_names:
        ds = ds.remove_columns(['token_type_ids'])

cols = [c for c in ['input_ids', 'attention_mask', 'label'] if c in train_ds.column_names]
train_ds.set_format(type='torch', columns=cols)
val_ds.set_format(type='torch', columns=cols)
test_ds.set_format(type='torch', columns=cols)


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/49188 [00:00<?, ? examples/s]

Map:   0%|          | 0/6149 [00:00<?, ? examples/s]

Map:   0%|          | 0/6149 [00:00<?, ? examples/s]

## 7. Fine-tune DistilBERT

Class weights compensate for the ~9:1 imbalance so the model doesn't just predict "clean"
for everything.


In [8]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch.nn as nn

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

n_pos = train_df['is_toxic'].sum()
n_neg = len(train_df) - n_pos
class_weights = torch.tensor([1.0, n_neg / n_pos], dtype=torch.float).to(device)
print("Class weights [clean, toxic]:", class_weights)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Class weights [clean, toxic]: tensor([ 1.0000, 10.6947], device='cuda:0')


In [11]:
import evaluate as hf_evaluate

precision_metric = hf_evaluate.load("precision")
recall_metric = hf_evaluate.load("recall")
f1_metric = hf_evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "precision": precision_metric.compute(predictions=preds, references=labels)["precision"],
        "recall": recall_metric.compute(predictions=preds, references=labels)["recall"],
        "f1": f1_metric.compute(predictions=preds, references=labels)["f1"],
    }

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.281400,0.314450,0.670947,0.794677,0.727589
2,0.204700,0.325642,0.673879,0.828897,0.743393
3,0.138400,0.454991,0.753176,0.788973,0.770659


TrainOutput(global_step=4614, training_loss=0.22763448576416656, metrics={'train_runtime': 529.1138, 'train_samples_per_second': 278.889, 'train_steps_per_second': 8.72, 'total_flos': 4886854803818496.0, 'train_loss': 0.22763448576416656, 'epoch': 3.0})

In [12]:
import os
print(os.path.exists("./results"))
if os.path.exists("./results"):
    print(os.listdir("./results"))

True
['checkpoint-4614', 'checkpoint-1538', 'checkpoint-3076']


### CHECKPOINT — save the trained model to Drive now

Do this immediately after training finishes, before touching anything else. If Colab
disconnects later, you reload from here instead of retraining (training is the slowest step).


In [13]:
SAVE_DIR = os.path.join(DRIVE_DIR, 'model')
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Model checkpoint saved to", SAVE_DIR)


Model checkpoint saved to /content/drive/MyDrive/toxicity-project/model


---
**If you're resuming from here after a disconnect**, run this cell instead of everything
above, then skip to Section 8:

```python
import os, json, torch
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/toxicity-project'
SAVE_DIR = os.path.join(DRIVE_DIR, 'model')

from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)

import pandas as pd
test_df = pd.read_csv(os.path.join(DRIVE_DIR, 'test_set.csv'))
```
---


## 8. Evaluate honestly — report the real precision/recall trade-off

No target number is assumed here. Instead we compute probabilities on the held-out test set
(data never used in training) and show the actual trade-off across several thresholds, so you
can pick one based on real numbers, not a number decided in advance.


In [14]:
from sklearn.metrics import precision_recall_curve
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification

eval_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(device).eval()

test_inputs = tokenizer(test_df['comment_text'].tolist(), truncation=True, padding=True,
                         max_length=128, return_tensors="pt")
test_inputs.pop("token_type_ids", None)
labels = test_df['is_toxic'].values

all_probs = []
batch_size = 64
with torch.no_grad():
    for i in range(0, len(test_df), batch_size):
        batch = {k: v[i:i+batch_size].to(device) for k, v in test_inputs.items()}
        logits = eval_model(**batch).logits
        probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        all_probs.extend(probs.tolist())

probs = np.array(all_probs)
precisions, recalls, thresholds = precision_recall_curve(labels, probs)

print(f"Computed real precision/recall curve on {len(test_df):,} held-out test examples")


Computed real precision/recall curve on 6,149 held-out test examples


In [15]:
# Show the real trade-off at a spread of thresholds — nothing hardcoded or targeted
f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-9)
best_f1_idx = np.argmax(f1_scores)

print(f"{'Threshold':>10} | {'Precision':>10} | {'Recall':>10}")
print("-" * 38)
for t in [0.3, 0.5, 0.7, 0.8, 0.9, 0.95]:
    idx = np.searchsorted(thresholds, t)
    idx = min(idx, len(precisions) - 2)
    print(f"{thresholds[idx]:>10.3f} | {precisions[idx]:>9.1%} | {recalls[idx]:>9.1%}")

print()
print(f"Best-F1 balance point (no target imposed):")
print(f"  Threshold: {thresholds[best_f1_idx]:.3f}")
print(f"  Precision: {precisions[:-1][best_f1_idx]:.1%}")
print(f"  Recall:    {recalls[:-1][best_f1_idx]:.1%}")
print(f"  F1:        {f1_scores[best_f1_idx]:.1%}")

# Default to the best-F1 threshold as the production choice.
# Change this manually if you want a different point on the trade-off curve above.
best_threshold = float(thresholds[best_f1_idx])
final_precision = float(precisions[:-1][best_f1_idx])
final_recall = float(recalls[:-1][best_f1_idx])
final_f1 = float(f1_scores[best_f1_idx])

print()
print(f"Using threshold {best_threshold:.3f} going forward "
      f"(precision {final_precision:.1%}, recall {final_recall:.1%})")


 Threshold |  Precision |     Recall
--------------------------------------
     0.301 |     71.1% |     80.0%
     0.502 |     74.5% |     78.1%
     0.710 |     77.2% |     77.4%
     0.801 |     78.3% |     76.0%
     0.900 |     80.5% |     74.5%
     0.951 |     83.2% |     72.4%

Best-F1 balance point (no target imposed):
  Threshold: 0.734
  Precision: 77.8%
  Recall:    77.4%
  F1:        77.6%

Using threshold 0.734 going forward (precision 77.8%, recall 77.4%)


In [16]:
# Persist the chosen threshold + real metrics to Drive
with open(os.path.join(SAVE_DIR, "inference_config.json"), "w") as f:
    json.dump({
        "decision_threshold": best_threshold,
        "precision": final_precision,
        "recall": final_recall,
        "f1": final_f1,
    }, f)
print("Saved threshold + metrics to", SAVE_DIR)


Saved threshold + metrics to /content/drive/MyDrive/toxicity-project/model


## 9. Export to ONNX

Uses `dynamo=False` — the older, stable export path. The newer default exporter splits
model weights into a separate `.data` file, which breaks quantization; `dynamo=False` keeps
everything in one `.onnx` file.


In [17]:
ONNX_DIR = os.path.join(DRIVE_DIR, 'onnx')
os.makedirs(ONNX_DIR, exist_ok=True)

model_for_export = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to("cpu").eval()

dummy_inputs = tokenizer("This is a sample sentence for export.", return_tensors="pt",
                          truncation=True, max_length=128, padding="max_length")
dummy_inputs.pop("token_type_ids", None)

torch.onnx.export(
    model_for_export,
    (dummy_inputs["input_ids"], dummy_inputs["attention_mask"]),
    os.path.join(ONNX_DIR, "model.onnx"),
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch", 1: "sequence"},
        "attention_mask": {0: "batch", 1: "sequence"},
        "logits": {0: "batch"},
    },
    opset_version=17,
    dynamo=False,
)
tokenizer.save_pretrained(ONNX_DIR)
print("Exported fp32 ONNX model to", ONNX_DIR)


/tmp/ipykernel_3750/2232202859.py:10: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Exported fp32 ONNX model to /content/drive/MyDrive/toxicity-project/onnx


## 10. Quantize to INT8

Plain `onnxruntime.quantization.quantize_dynamic` — no `optimum`, no preprocessing step
(both caused failures during development). This is the simplest path that reliably works.


In [18]:
from onnxruntime.quantization import quantize_dynamic, QuantType

QUANT_DIR = os.path.join(DRIVE_DIR, 'onnx-quantized')
os.makedirs(QUANT_DIR, exist_ok=True)

quantize_dynamic(
    model_input=os.path.join(ONNX_DIR, "model.onnx"),
    model_output=os.path.join(QUANT_DIR, "model_quantized.onnx"),
    weight_type=QuantType.QInt8,
)
tokenizer.save_pretrained(QUANT_DIR)
print("Quantized model saved to", QUANT_DIR)


Quantized model saved to /content/drive/MyDrive/toxicity-project/onnx-quantized


## 11. Latency benchmark

PyTorch eager vs ONNX fp32 vs ONNX int8, all on CPU — the realistic serving target for
this workload (GPU inference on single short sequences is dominated by launch overhead,
not compute).


In [19]:
import onnxruntime as ort
from transformers import AutoModelForSequenceClassification as _AMFSC

N_WARMUP = 10
N_RUNS = 100
SAMPLE_TEXT = "This is a sample post that needs to be checked for policy violations."

def benchmark_pytorch(model_dir, text, n_warmup=N_WARMUP, n_runs=N_RUNS):
    tok = AutoTokenizer.from_pretrained(model_dir)
    mdl = _AMFSC.from_pretrained(model_dir).to("cpu").eval()
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=128, padding="max_length")
    inputs.pop("token_type_ids", None)
    with torch.no_grad():
        for _ in range(n_warmup):
            mdl(**inputs)
        times = []
        for _ in range(n_runs):
            t0 = time.perf_counter()
            mdl(**inputs)
            times.append((time.perf_counter() - t0) * 1000)
    return np.array(times)

def benchmark_onnx(model_dir, text, n_warmup=N_WARMUP, n_runs=N_RUNS):
    tok = AutoTokenizer.from_pretrained(model_dir)
    onnx_filename = "model_quantized.onnx" if os.path.exists(os.path.join(model_dir, "model_quantized.onnx")) else "model.onnx"
    session = ort.InferenceSession(os.path.join(model_dir, onnx_filename))
    inputs = tok(text, return_tensors="np", truncation=True, max_length=128, padding="max_length")
    onnx_inputs = {k: v for k, v in inputs.items() if k in [i.name for i in session.get_inputs()]}
    for _ in range(n_warmup):
        session.run(None, onnx_inputs)
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        session.run(None, onnx_inputs)
        times.append((time.perf_counter() - t0) * 1000)
    return np.array(times)

pt_times = benchmark_pytorch(SAVE_DIR, SAMPLE_TEXT)
onnx_fp32_times = benchmark_onnx(ONNX_DIR, SAMPLE_TEXT)
onnx_int8_times = benchmark_onnx(QUANT_DIR, SAMPLE_TEXT)

def summarize(name, times):
    print(f"{name:25s} | mean {times.mean():6.2f}ms | p50 {np.median(times):6.2f}ms | p95 {np.percentile(times,95):6.2f}ms")

summarize("PyTorch (CPU)", pt_times)
summarize("ONNX Runtime (fp32)", onnx_fp32_times)
summarize("ONNX Runtime (int8)", onnx_int8_times)


PyTorch (CPU)             | mean 154.93ms | p50 131.41ms | p95 290.83ms
ONNX Runtime (fp32)       | mean 131.63ms | p50 118.89ms | p95 182.35ms
ONNX Runtime (int8)       | mean  84.15ms | p50  82.77ms | p95  92.94ms


## 12. Final summary — your real, actual numbers

In [20]:
print("=" * 55)
print("RESULTS SUMMARY (all real, measured numbers)")
print("=" * 55)
print(f"Dataset: {len(df):,} labeled posts")
print(f"  - Jigsaw (Wikipedia comments): {(df['source']=='jigsaw_comments').sum():,}")
print(f"  - Twitter (real tweets):        {(df['source']=='twitter_posts').sum():,}")
print(f"Model: DistilBERT (fine-tuned, {MODEL_NAME})")
print(f"Decision threshold: {best_threshold:.3f} (best-F1 balance point)")
print(f"Precision: {final_precision:.1%}")
print(f"Recall:    {final_recall:.1%}")
print(f"F1:        {final_f1:.1%}")
print(f"Latency (ONNX int8, p50): {np.median(onnx_int8_times):.1f}ms")
print(f"Latency (ONNX int8, p95): {np.percentile(onnx_int8_times,95):.1f}ms")
print(f"Speedup vs PyTorch eager: {pt_times.mean() / onnx_int8_times.mean():.1f}x")


RESULTS SUMMARY (all real, measured numbers)
Dataset: 61,486 labeled posts
  - Jigsaw (Wikipedia comments): 31,959
  - Twitter (real tweets):        29,527
Model: DistilBERT (fine-tuned, distilbert-base-uncased)
Decision threshold: 0.734 (best-F1 balance point)
Precision: 77.8%
Recall:    77.4%
F1:        77.6%
Latency (ONNX int8, p50): 82.8ms
Latency (ONNX int8, p95): 92.9ms
Speedup vs PyTorch eager: 1.8x


## 13. Try it live

In [21]:
def predict_toxicity(text, threshold=None):
    session = ort.InferenceSession(os.path.join(QUANT_DIR, "model_quantized.onnx"))
    tok = AutoTokenizer.from_pretrained(QUANT_DIR)
    if threshold is None:
        threshold = best_threshold

    inputs = tok(text, return_tensors="np", truncation=True, max_length=128, padding="max_length")
    onnx_inputs = {k: v for k, v in inputs.items() if k in [i.name for i in session.get_inputs()]}
    logits = session.run(None, onnx_inputs)[0]
    exp_logits = np.exp(logits[0] - logits[0].max())
    prob_toxic = float((exp_logits / exp_logits.sum())[1])
    return {
        "text": text,
        "toxic_probability": prob_toxic,
        "flagged": prob_toxic >= threshold,
    }

print(predict_toxicity("Have a great day everyone!"))
print(predict_toxicity("You are a worthless idiot and I hope something bad happens to you"))


{'text': 'Have a great day everyone!', 'toxic_probability': 0.0020027155987918377, 'flagged': False}
{'text': 'You are a worthless idiot and I hope something bad happens to you', 'toxic_probability': 0.9991430044174194, 'flagged': True}


## 14. Serve it — FastAPI REST API

Wraps the quantized ONNX model in a real FastAPI endpoint. Uses the `tokenizer` and
`QUANT_DIR` / `best_threshold` already loaded earlier in this notebook — no re-download
needed if you're continuing in the same session.


In [22]:
!pip install -q fastapi uvicorn pyngrok

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title="Content Moderation API")

# CORS enabled from the start — needed for the browser demo in section 14c to work
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class PredictRequest(BaseModel):
    text: str

class PredictResponse(BaseModel):
    text: str
    toxic_probability: float
    flagged: bool
    threshold_used: float

def run_inference(text: str) -> dict:
    session = ort.InferenceSession(os.path.join(QUANT_DIR, "model_quantized.onnx"))
    inputs = tokenizer(text, return_tensors="np", truncation=True, max_length=128, padding="max_length")
    onnx_inputs = {k: v for k, v in inputs.items() if k in [i.name for i in session.get_inputs()]}
    logits = session.run(None, onnx_inputs)[0]
    exp_logits = np.exp(logits[0] - logits[0].max())
    prob_toxic = float((exp_logits / exp_logits.sum())[1])
    return {
        "text": text,
        "toxic_probability": prob_toxic,
        "flagged": prob_toxic >= best_threshold,
        "threshold_used": best_threshold,
    }

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictResponse)
def predict(request: PredictRequest):
    if not request.text.strip():
        raise HTTPException(status_code=400, detail="text must not be empty")
    return run_inference(request.text)

print("FastAPI app defined, CORS enabled.")


FastAPI app defined, CORS enabled.


### 14a. Test it without a live server (proves the API code works)

In [23]:
from fastapi.testclient import TestClient
import onnxruntime as ort
client = TestClient(app)

response1 = client.post("/predict", json={"text": "Have a great day everyone!"})
response2 = client.post("/predict", json={"text": "You are a worthless idiot and I hope something bad happens to you"})

print("Response 1:", response1.json())
print("Response 2:", response2.json())


Response 1: {'text': 'Have a great day everyone!', 'toxic_probability': 0.0020027155987918377, 'flagged': False, 'threshold_used': 0.7341927289962769}
Response 2: {'text': 'You are a worthless idiot and I hope something bad happens to you', 'toxic_probability': 0.9991430044174194, 'flagged': True, 'threshold_used': 0.7341927289962769}


### 14b. (Optional) Make it a live public URL

Needs a free ngrok account: https://dashboard.ngrok.com/signup, then your authtoken from
https://dashboard.ngrok.com/get-started/your-authtoken

**Run this cell only ONCE per session.** Re-running it after the server already started
causes a port conflict (old server threads can't be killed, only a full Colab restart
clears them) — if that happens, Runtime → Restart session, re-run from Section 14 onward
(not from the very top — your model is safe on Drive), and run this cell once.


In [24]:
from pyngrok import ngrok
import nest_asyncio, uvicorn, threading

NGROK_AUTHTOKEN = "PASTE_YOUR_TOKEN_HERE"
ngrok.set_auth_token(NGROK_AUTHTOKEN)

nest_asyncio.apply()
ngrok.kill()  # close any previous tunnels first

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_server, daemon=True).start()
public_url = ngrok.connect(8000)
print("Your API is live at:", str(public_url))
print("Interactive docs:", str(public_url) + "/docs")


INFO:     Started server process [3750]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Your API is live at: NgrokTunnel: "https://sporting-dazzler-nurture.ngrok-free.dev" -> "http://localhost:8000"
Interactive docs: NgrokTunnel: "https://sporting-dazzler-nurture.ngrok-free.dev" -> "http://localhost:8000"/docs


### 14c. (Optional) Simple HTML demo page

Paste the URL printed by 14b into `YOUR_NGROK_URL` below. Includes the
`ngrok-skip-browser-warning` header — without it, free ngrok URLs show an interstitial
warning page that silently breaks `fetch()` calls from a browser.


In [27]:
from IPython.display import HTML

YOUR_NGROK_URL = "https://sporting-dazzler-nurture.ngrok-free.dev"  # e.g. "https://xxxx.ngrok-free.dev"

demo_html = f"""
<div style="font-family: sans-serif; max-width: 500px; padding: 20px; border: 1px solid #ccc; border-radius: 8px;">
  <h3>Content Moderation Demo</h3>
  <textarea id="inputText" rows="3" style="width: 100%; padding: 8px;" placeholder="Type a post or comment..."></textarea>
  <button onclick="checkText()" style="margin-top: 10px; padding: 8px 16px;">Check Toxicity</button>
  <div id="result" style="margin-top: 15px; font-weight: bold;"></div>
</div>

<script>
async function checkText() {{
  const text = document.getElementById('inputText').value;
  const resultDiv = document.getElementById('result');
  resultDiv.innerText = 'Checking...';
  try {{
    const response = await fetch('{YOUR_NGROK_URL}/predict', {{
      method: 'POST',
      headers: {{
        'Content-Type': 'application/json',
        'ngrok-skip-browser-warning': 'true'
      }},
      body: JSON.stringify({{text: text}})
    }});
    const data = await response.json();
    const pct = (data.toxic_probability * 100).toFixed(1);
    resultDiv.innerHTML = data.flagged
      ? `<span style="color: red;">FLAGGED — ${{pct}}% toxic probability</span>`
      : `<span style="color: green;">Not flagged — ${{pct}}% toxic probability</span>`;
  }} catch (e) {{
    resultDiv.innerText = 'Error: ' + e.message;
  }}
}}
</script>
"""

HTML(demo_html)
